# ARC-AGI-2 — Kaggle submission (Qwen3-8B 4-bit)

**Setup**
1. Add this repo as a Kaggle Dataset (or clone via internet-enabled session, then Save Version offline).
2. Add competition data `arc-prize-2025`.
3. Optionally add a HF model dataset mirror of `Qwen/Qwen3-8B` for offline scoring.
4. Accelerator: GPU (P100 / T4). Internet **off** for final submit.

Writes `/kaggle/working/submission.json` in ARC Prize format.

In [ ]:
import os, sys, subprocess, pathlib

# Locate repo: either attached Kaggle dataset or /kaggle/working checkout
CANDIDATES = [
    pathlib.Path('/kaggle/input/arc-solver'),
    pathlib.Path('/kaggle/input/puzzle-solver-arc-solver'),
    pathlib.Path('/kaggle/working/arc-solver'),
    pathlib.Path('.').resolve(),
]
ROOT = None
for p in CANDIDATES:
    if (p / 'src' / 'pipeline.py').exists():
        ROOT = p
        break
assert ROOT is not None, 'arc-solver repo not found — attach as Kaggle dataset'
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('ROOT', ROOT)

# Ensure runtime deps (no-op if already present on the image)
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'bitsandbytes>=0.43.0', 'accelerate>=0.33.0', 'transformers>=4.51.0', 'pyyaml',
])

In [ ]:
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/1e9)

In [ ]:
# Smoke: load + one short generate (skip on final long run by setting RUN_SMOKE=False)
RUN_SMOKE = True
if RUN_SMOKE:
    from src.llm_client import LLMClient
    client = LLMClient(
        backend='kaggle_local',
        model_name=os.environ.get('ARC_MODEL', 'Qwen/Qwen3-8B'),
        load_in_4bit=True,
        device_map='auto',
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype='float16',
        bnb_4bit_use_double_quant=True,
    )
    print(repr(client.generate('Reply with exactly: PONG', max_tokens=16, temperature=0.0)[:200]))
    del client
    import gc; gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Full submission build — uses config.kaggle.yaml
import subprocess, sys
cmd = [
    sys.executable, '-m', 'scripts.build_submission',
    '--config', str(ROOT / 'config.kaggle.yaml'),
    '--out', '/kaggle/working/submission.json',
    # Uncomment to dry-run a few tasks first:
    # '--limit', '3',
]
print('Running', cmd)
subprocess.check_call(cmd)

In [ ]:
import json, pathlib
p = pathlib.Path('/kaggle/working/submission.json')
data = json.loads(p.read_text())
print('tasks', len(data), 'file_mb', round(p.stat().st_size/1e6, 2))
tid = next(iter(data))
print('sample', tid, data[tid][0].keys())